## Purge Zero-Works Author Records

The `authors_for_matching` pipeline fix (walden `c5e905b`, 2026-05-03) filters zero-works profiles out of the match-candidate pool, but the underlying rows still live in `openalex.authors.authors` and Elasticsearch. As of 2026-05-03 sizing this is **3,810,728 profiles** (~3.17% of the table):

| Source | Count |
|---|---|
| Drained merge losers (Phases 1/2/3b/3c/4/4b-c/5) | ~879K |
| Legacy MAG/Crossref junk (no created_date, no full_name, no affiliations) | ~2.93M |

This notebook **DELETEs** every author profile where `works_count = 0` from `openalex.authors.authors`. A live recheck against `openalex.works.work_authors` skips any profile that has accumulated a race-tail row since the snapshot was taken. Full rollback JSON is captured in `openalex.authors.zero_works_purge_log`.

**Out of scope:** ES sync. After this notebook completes and `CreateAuthors` rebuilds `openalex_authors`, the deleted IDs need to be reindexed into the new authors index separately. Stable-ID 301 redirect for merge-loser IDs already lives in the per-phase `author_merge_log_*` tables; legacy junk IDs will 404.

**Run order:** cells 1–6 are read-only snapshots and validations. Cells 7–8 are destructive — verify the validated count in cell 6 before running. Then re-run `notebooks/authors/CreateAuthors.ipynb` to rebuild `openalex_authors` without the deleted profiles.

### Cell 1: Snapshot zero-works candidates

`openalex_authors` is the materialized rollup that carries `works_count`. Snapshot the candidates plus the full source row from `openalex.authors.authors` for rollback.

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.zero_works_purge_snapshot AS
SELECT
  oa.id AS author_id,
  oa.display_name AS previous_display_name,
  oa.full_name AS previous_full_name,
  oa.orcid AS previous_orcid,
  oa.works_count AS previous_works_count,
  oa.raw_author_names AS previous_raw_author_names,
  TO_JSON(STRUCT(a.*)) AS previous_authors_row_json,
  current_timestamp() AS snapshot_timestamp
FROM openalex.authors.openalex_authors oa
JOIN openalex.authors.authors a ON a.id = oa.id
WHERE oa.works_count = 0

### Cell 2: Snapshot validation

`profiles_with_orcid` is a smell check — zero-works + ORCID = potential parser bug or mid-merge state worth eyeballing before purging. Drained Phase 1–5 merge losers will appear here; legacy MAG/Crossref junk dominates the rest.

In [ ]:
SELECT
  COUNT(*) AS profiles_to_delete,
  SUM(CASE WHEN previous_orcid IS NOT NULL AND previous_orcid <> '' THEN 1 ELSE 0 END) AS profiles_with_orcid,
  SUM(CASE WHEN previous_full_name IS NULL OR previous_full_name = '' THEN 1 ELSE 0 END) AS profiles_null_full_name,
  SUM(CASE WHEN previous_raw_author_names IS NULL OR SIZE(previous_raw_author_names) = 0 THEN 1 ELSE 0 END) AS profiles_no_raw_names
FROM openalex.authors.zero_works_purge_snapshot

### Cell 3: Live recheck filter

`openalex_authors.works_count` lags live `work_authors` by however long since the last `CreateAuthors` rebuild. Re-derive `works_count` against the live table and drop any candidate that has accumulated a row since the snapshot. The remaining set is what gets DELETEd.

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.zero_works_purge_validated AS
SELECT s.*
FROM openalex.authors.zero_works_purge_snapshot s
LEFT ANTI JOIN (
  SELECT DISTINCT author_id
  FROM openalex.works.work_authors
  WHERE author_id IS NOT NULL
) live ON live.author_id = s.author_id

### Cell 4: Live-recheck validation

`would_delete` is the final DELETE volume. `dropped_by_recheck` (= snapshot_size − would_delete) is the race-tail count that the live filter saved from a wrong delete.

In [ ]:
SELECT
  (SELECT COUNT(*) FROM openalex.authors.zero_works_purge_snapshot) AS snapshot_size,
  (SELECT COUNT(*) FROM openalex.authors.zero_works_purge_validated) AS would_delete,
  (SELECT COUNT(*) FROM openalex.authors.zero_works_purge_snapshot)
    - (SELECT COUNT(*) FROM openalex.authors.zero_works_purge_validated) AS dropped_by_recheck

### Cell 5: Audit log (for rollback)

Captures the full source row from `openalex.authors.authors` as JSON for every profile that will be deleted. To restore: parse `previous_value_json` and INSERT back row-by-row (column mapping is required because `authors` has many columns).

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.zero_works_purge_log AS
SELECT
  'author_profile' AS scope,
  CAST(author_id AS STRING) AS key1,
  previous_full_name AS previous_value_text,
  previous_authors_row_json AS previous_value_json,
  current_timestamp() AS purge_timestamp
FROM openalex.authors.zero_works_purge_validated

### Cell 6: Audit-log validation

Should equal `would_delete` from cell 4.

In [ ]:
SELECT COUNT(*) AS audit_log_rows FROM openalex.authors.zero_works_purge_log

### Cell 7: EXECUTE — DELETE zero-works profiles from openalex.authors.authors

**WARNING**: Destructive. Verify the count in cell 4 (`would_delete`) before running. The MERGE is gated on the validated set so any race-tail profile that grew works between cells 1 and 3 is skipped automatically.

In [ ]:
MERGE INTO openalex.authors.authors AS target
USING openalex.authors.zero_works_purge_validated AS source
  ON target.id = source.author_id
WHEN MATCHED THEN DELETE

### Cell 8: Post-cleanup verification

`profiles_remaining_from_log` should be 0 — no audit-log id should still resolve in `openalex.authors.authors`. `audit_log_rows` should equal `would_delete` from cell 4.

In [ ]:
SELECT 'profiles_remaining_from_log' AS check_name, COUNT(*) AS remaining
FROM openalex.authors.zero_works_purge_log l
JOIN openalex.authors.authors a ON a.id = l.key1
UNION ALL
SELECT 'audit_log_rows' AS check_name, COUNT(*) AS remaining
FROM openalex.authors.zero_works_purge_log

### Cell 9: Rollback template (use only if a systematic FP is discovered)

```sql
-- Restore deleted author profiles. previous_value_json holds the full row from
-- openalex.authors.authors at snapshot time — extract each column explicitly.
-- Add a WHERE clause to scope the rollback to a specific class if needed.
INSERT INTO openalex.authors.authors
SELECT FROM_JSON(previous_value_json, 'STRUCT<...>').*  -- fill in the schema
FROM openalex.authors.zero_works_purge_log;
```

### Downstream rebuild

After cell 7:
1. Re-run `notebooks/authors/CreateAuthors.ipynb` so `openalex_authors` and `authors_for_matching` drop the deleted IDs.
2. Reindex `openalex_authors` into the new ES authors index.
3. Stable-ID redirect: merge-loser IDs are already covered by per-phase `author_merge_log_*` tables. Legacy junk IDs will 404 — acceptable per scope decision.